In [4]:
import numpy as np

In [7]:
import matplotlib.pyplot as plt

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    from numba import jit
except ModuleNotFoundError:
    # Fallback: si numba no está disponible, jit no altera la función.
    def jit(*args, **kwargs):
        def deco(func):
            return func
        return deco

from typing import Tuple, List, Dict, Any

In [ ]:
class Sistema:

    def __init__(self, particles: List[Dict[str, Any]]):

        '''
        Clase que representa un sistema general de partículas, que pueden ser núcleos masivos o estrellas de masa cero.
        params:
        - particles: lista de diccionarios con la información de las partículas (núcleos y estrellas)
            - Cada partícula es un diccionario con las siguientes claves:
                - 'mass': masa de la partícula (0 para estrellas)
                - 'position': posición inicial (array de 3 elementos)
                - 'velocity': velocidad inicial (array de 3 elementos)
        '''
        if particles is None or not isinstance(particles, list):
            print("No se proporcionó una lista válida de partículas.\n\
                  Se inicializa el sistema sin partículas.")
            self.particles = []
        else:
            self.particles = particles

        ## Masa del sistema, es la suma de las masas de todas las partículas
        self.mass = sum(particle['mass'] for particle in particles)
        # Número de partículas, posiciones y velocidades iniciales
        self.num_particles = len(particles)
        # ïndices de partículas con masa
        self.mass_indices = np.where(np.array([particle['mass'] for particle in particles]) > 0)[0]

        # Almacenamos las posiciones y velocidades iniciales en arrays para facilitar los cálculos posteriores
        self.masses = np.array([particle['mass'] for particle in particles])
        self.positions = np.array([particle['position'] for particle in particles])
        self.velocities = np.array([particle['velocity'] for particle in particles])


        ## tiempo de relajación del sistema
        v_prom = np.mean(np.linalg.norm(self.velocities, axis=1))
        n = self.num_particles
        self.t_relax = (n / (8 * np.log(n))) * (self.mass / v_prom**3) if v_prom > 0 else np.inf


    # Un método estático (@staticmethod) no recibe la instancia de la clase ('self')
    # como argumento y funciona realmente como una función "independiente" agrupada aquí.
    # Lo usamos porque Numba (@jit / @njit) solo compila tipos matemáticos de NumPy y
    # primitivos nativos; si a Numba le pasáramos 'self' o atributos con diccionarios
    # y listas (como self.particles), fallaría al intentar compilarlos.
    @staticmethod
    @jit(nopython=True)  # @njit es simplemente un alias corto para @jit(nopython=True)
    def get_accelerations(positions, masses, massive_indices, num_particles):
        '''
        Calcula simultáneamente las aceleraciones de todas las partículas bajo el
        influjo de los núcleos masivos, optimizado a nivel de C con Numba.

        $$a = \sum_{k \in massive_indices} G * m_k * (r_i - r_k) / |r_i - r_k|^3$$

        parametros:
        - positions: array de posiciones de las partículas (num_particles x 3)
        - masses: array de masas de las partículas (num_particles)
        - massive_indices: array de índices de las partículas que son núcleos masivos
        - num_particles: número total de partículas en el sistema

        '''
        a = np.zeros((num_particles, 3), dtype=np.float64)
        for i in range(num_particles):
            for k in massive_indices:
                if i != k:
                    dx = positions[i, 0] - positions[k, 0]
                    dy = positions[i, 1] - positions[k, 1]
                    dz = positions[i, 2] - positions[k, 2]
                    r_mag = np.sqrt(dx**2 + dy**2 + dz**2)
                    if r_mag > 0:
                        factor = masses[k] / (r_mag**3)
                        a[i, 0] -= factor * dx
                        a[i, 1] -= factor * dy
                        a[i, 2] -= factor * dz
        return a

    def integrate(self, dt: float, T: float,
                  actualizar: bool = False
                  )-> np.ndarray:
        '''Integramos el problema usando leapfrog

        Necesitamos las posiciones y velocidades iniciales de las partículas, así como sus masas.
        Luego, en cada paso de tiempo, calculamos las fuerzas gravitacionales entre los núcleos masivos y las estrellas
        (teniendo en cuenta que las estrellas no interactúan entre sí), actualizamos las velocidades de las estrellas y
        luego sus posiciones.
        '''

        num_steps = int(T / dt)

        trajectory = np.zeros((num_steps, self.num_particles, 3))

        # 1. astype(np.float64): Numba es de tipado estricto. Si le pasamos enteros mezclados con flotantes,
        # la compilación puede fallar. Aseguramos que todo sea float64 para máxima compatibilidad.
        # 2. copy=True: En NumPy, 'a = b' solo crea una referencia. Si más abajo hacemos 'a += 1', 'b' también se modifica.
        # Creamos copias explícitas aislando la memoria, así las variables de la clase (self.xxx) no se modifican
        # accidentalmente durante la simulación (útil si actualizar=False).
        positions = self.positions.astype(np.float64, copy=True)
        velocities = self.velocities.astype(np.float64, copy=True)
        masses = self.masses.astype(np.float64)

        trajectory[0] = positions

        massive_indices = self.mass_indices

        # Calculamos aceleraciones iniciales llamando a la función JIT compilada
        a_i = self.get_accelerations(positions, masses, massive_indices, self.num_particles)
        velocities += 0.5 * a_i * dt  # Medio paso de leapfrog iterando rápido

        # Iteramos sobre cada paso usando la Numba JIT para las aceleraciones
        for step in range(1, num_steps):
            positions += velocities * dt
            trajectory[step] = positions

            # Nueva aceleración
            a_i = self.get_accelerations(positions, masses, massive_indices, self.num_particles)
            velocities += a_i * dt

        if actualizar:
            self.positions = positions
            self.velocities = velocities
            # Sincronizamos las partículas para no dejar inconsistente los diccionarios
            for i in range(self.num_particles):
                self.particles[i]['position'] = positions[i]
                self.particles[i]['velocity'] = velocities[i]

        return trajectory

    # Cuadraturas del sistema

    ## Momento angular total del sistema
    def angular_momentum(self):
        '''Calcula el momento angular total del sistema, que es la suma del momento angular de cada partícula.

        El momento angular de una partícula se calcula como L = r x (m * v), donde r es la posición de la partícula,
        m es su masa y v es su velocidad. Para las estrellas, m=0, por lo que no contribuyen al momento angular.
        '''
        L_total = np.zeros(3)
        for particle in self.particles:
            if particle['mass'] > 0:  # Solo consideramos el momento angular de los núcleos masivos
                r = particle['position']
                v = particle['velocity']
                m = particle['mass']
                L_total += np.cross(r, m * v)
        return L_total


    ## Energía total del sistema
    def total_energy(self):
        '''Calcula la energía total del sistema, que es la suma de la energía cinética y la energía potencial gravitatoria.

        La energía cinética de una partícula se calcula como K = 0.5 * m * v^2, donde m es la masa de la partícula
        y v es su velocidad. Para las estrellas, m=0, por lo que no contribuyen a la energía cinética.

        La energía potencial gravitatoria entre dos partículas se calcula como U = -G * m1 * m2 / r, donde G es la
        constante gravitacional (que asumimos G=1), m1 y m2 son las masas de las partículas y r es la distancia entre ellas.
        Solo consideramos la energía potencial entre los núcleos masivos, ya que las estrellas no interactúan entre sí.
        '''
        K_total = 0.0
        U_total = 0.0

        # Calcular energía cinética
        for particle in self.mass_indices:
            m = particle['mass']
            v = particle['velocity']
            K_total += 0.5 * m * np.dot(v, v)
        # Calcular energía potencial entre núcleos masivos
        massive_particles = [p for p in self.particles if p['mass'] > 0]
        for i in range(len(massive_particles)):
            for j in range(i + 1, len(massive_particles)):
                m1 = massive_particles[i]['mass']
                m2 = massive_particles[j]['mass']
                r_vec = massive_particles[i]['position'] - massive_particles[j]['position']
                r_mag = np.linalg.norm(r_vec)
                if r_mag > 0:
                    U_total -= (m1 * m2) / r_mag  # G=1

        return K_total + U_total

    ## plot posiciones de las partículas en 2D
    def plot_positions(self):
        '''Genera un gráfico 2D (plano XY) de las posiciones de las partículas en el sistema.
        Los núcleos masivos se representan con un color diferente a las estrellas de masa cero.
        '''
        plt.figure(figsize=(7, 7))

        # Obtener las masas para filtrar índices
        masses = self.masses
        massive_idx = self.mass_indices
        stars_idx = np.where(masses == 0)[0]

        # Graficar usando self.positions directamente
        plt.scatter(self.positions[stars_idx, 0], self.positions[stars_idx, 1],
                    s=3, c='blue', label='Estrellas de masa cero')
        plt.scatter(self.positions[massive_idx, 0], self.positions[massive_idx, 1],
                    s=50, c='red', label='Núcleo masivo')

        plt.title('Posiciones de las partículas en el sistema (2D)')
        plt.xlabel('X')
        plt.ylabel('Y')
        plt.axis('equal')
        plt.legend()
        plt.show()


    def animate_trajectories(self, trayectorias = None, actualizar: bool = False,
                             dt = None, T = None, step_skip: int = 1,
                             limite_ejes: float = None):
        '''
        Genera una animación de las trayectorias de las partículas a lo largo del tiempo. En formato HTML5 para Jupyter Notebook.

        params:
        - trayectorias: array de forma (num_steps, num_particles, 3) con las posiciones de las partículas en cada paso de tiempo.
                    Si es None, se calcula la trayectoria usando el método integrate con los parámetros por defecto.
        - step_skip: int. Toma un fotograma cada cantidad de pasos especificada para reducir el peso de la animación.
        - limite_ejes: float. Establece el límite de los ejes para la animación.
        '''

        if trayectorias is None:
            try:
                trayectorias = self.integrate(dt=dt, T=T, actualizar=actualizar)
            except Exception as e:
                print(f"Error al calcular trayectorias: {e}")
                return

        # Filtramos la trayectoria según la cantidad de fotogramas a saltar
        if step_skip > 1:
            trayectorias = trayectorias[::step_skip]

        from matplotlib.animation import FuncAnimation
        from IPython.display import HTML
        fig, ax = plt.subplots(figsize=(7, 7))
        ax.set_title('Animación de trayectorias de las partículas')
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        if limite_ejes is None:
            # Si no se especifica es 1.5 veces el máximo radio de las posiciones iniciales para asegurar que se vean todas las partículas
            max_radius = np.max(np.linalg.norm(self.positions[:, :2], axis=1))
            limite_ejes = max_radius * 1.5
        ax.set_ylim(-limite_ejes, limite_ejes)
        ax.set_xlim(-limite_ejes, limite_ejes)
        ax.axis('equal')
        # Obtener las masas para filtrar índices
        masses = self.masses
        massive_idx = self.mass_indices
        stars_idx = np.where(masses == 0)[0]
        # Inicializar los puntos de las partículas
        points_stars, = ax.plot([], [], 'bo', markersize=3, label='Estrellas de masa cero')
        points_massive, = ax.plot([], [], 'ro', markersize=8, label='Núcleo masivo')
        ax.legend()
        def init():
            points_stars.set_data([], [])
            points_massive.set_data([], [])
            return points_stars, points_massive
        def update(frame):
            points_stars.set_data(trayectorias[frame, stars_idx, 0], trayectorias[frame, stars_idx, 1])
            points_massive.set_data(trayectorias[frame, massive_idx, 0], trayectorias[frame, massive_idx, 1])
            return points_stars, points_massive
        anim = FuncAnimation(fig, update, frames=trayectorias.shape[0], init_func=init, blit=True)
        plt.close(fig)  # Evitar mostrar la figura
        return HTML(anim.to_jshtml())

In [ ]:
particulas = [
    {"mass": 1e3, "position": np.array([0.0, 0.0, 0.0]), "velocity": np.array([0.0, 0.0, 0.0])}
]

## añadimos al rededor del núcleo de forma aleatoria 1000 estrellas de masa cero

for _ in range(1000):
    r = np.random.uniform(0.1, 10.0)  # Distancia aleatoria entre 0.1 y 10 unidades
    theta = np.random.uniform(0, 2 * np.pi)  # Ángulo azimutal aleatorio
    phi = np.random.uniform(0, np.pi)  # Ángulo polar aleatorio

    # Convertimos las coordenadas esféricas a cartesianas
    x = r * np.sin(phi) * np.cos(theta)
    y = r * np.sin(phi) * np.sin(theta)
    z = r * np.cos(phi)

    position = np.array([x, y, z])
    velocity = np.array([0.0, 0.0, 0.0])  # Velocidad inicial de las estrellas

    particulas.append({"mass": 0.0, "position": position, "velocity": velocity})


In [ ]:
class Galaxia(Sistema):
    '''Clase que representa una galaxia de disco según el modelo de Toomre.
    Hereda de la clase Sistema, pero con condiciones específicas para representar una galaxia de disco.
    '''
    def __init__(self, num_estrellas: int,
                 masa_nucleo: float,
                 radio_max: float,
                 radio_min: float = 0.5,
                 sentido_rotacion: int = 1):
        '''
        Inicializa una galaxia de disco con un núcleo masivo central y un conjunto de estrellas distribuidas en un disco alrededor del núcleo.

        params:
        - num_estrellas: número de estrellas (partículas de masa cero) en el disco
        - masa_nucleo: masa del núcleo masivo central
        - radio_max: radio máximo del disco de estrellas
        - radio_min: radio mínimo del disco de estrellas (default 0.5)
        - sentido_rotacion: 1 para anti-horario, -1 para horario.
        '''
        particles = []
        # Núcleo masivo central
        particles.append({"mass": masa_nucleo, "position": np.array([0.0, 0.0, 0.0]), "velocity": np.array([0.0, 0.0, 0.0])})

        # Estrellas distribuidas en un disco alrededor del núcleo
        for _ in range(num_estrellas):
            r = np.random.uniform(radio_min, radio_max)  # Distancia aleatoria entre radio_min y radio_max
            theta = np.random.uniform(0, 2 * np.pi)  # Ángulo aleatorio en el plano XY

            # Coordenadas cartesianas en el plano XY (disco)
            x = r * np.cos(theta)
            y = r * np.sin(theta)
            z = 0.0  # Las estrellas están en el plano del disco

            position = np.array([x, y, z])

            # Velocidad circular inicial para las estrellas: v = sqrt(M / r) asumiendo G=1
            v_mag = np.sqrt(masa_nucleo / r)

            # Velocidad perpendicular a la posición (-sin(theta), cos(theta)) multiplicada por el sentido de rotación
            vx = -sentido_rotacion * v_mag * np.sin(theta)
            vy = sentido_rotacion * v_mag * np.cos(theta)
            vz = 0.0

            velocity = np.array([vx, vy, vz])

            particles.append({"mass": 0.0, "position": position, "velocity": velocity})

        super().__init__(particles)

In [ ]:
galaxia1 = Galaxia(num_estrellas=1000, masa_nucleo=1e3,
                   radio_max=10.0, radio_min = 3)

In [ ]:
galaxia1.plot_positions()

In [ ]:
galaxia1.animate_trajectories(dt=0.01, T=6.0, step_skip=2)

In [ ]:
#ESTAAAAA
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

# Parámetros de simulación y animación
# Ajusta T, dt o step_skip para más/menos duración y fluidez.
dt = 0.01
T = 6.0
step_skip = 2
trail_length = 40

# Calculamos trayectorias de galaxia1 (integrate requiere dt y T)
tray_g1 = galaxia1.integrate(dt=dt, T=T, actualizar=False)
tray_g1 = tray_g1[::step_skip]

masses = galaxia1.masses
idx_estrellas = np.where(masses == 0)[0]
idx_nucleo = np.where(masses > 0)[0]

limite = 1.15 * np.max(np.linalg.norm(tray_g1[:, :, :2], axis=2))

fig, ax = plt.subplots(figsize=(7, 7))
ax.set_facecolor("black")
ax.set_title("Galaxia 1: partículas y trayectorias")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_xlim(-limite, limite)
ax.set_ylim(-limite, limite)
ax.set_aspect("equal")

# Partículas actuales
sc_estrellas = ax.scatter([], [], s=2, c="cyan", alpha=0.8, label="Estrellas")
sc_nucleo = ax.scatter([], [], s=70, c="gold", alpha=1.0, label="Núcleo")

# Rastros (trayectorias recientes)
sc_traza_estrellas = ax.scatter([], [], s=0.6, c="deepskyblue", alpha=0.18)
sc_traza_nucleo = ax.scatter([], [], s=10, c="orange", alpha=0.35)

ax.legend(loc="upper right")

def _empty_offsets():
    return np.empty((0, 2), dtype=float)

def init():
    sc_estrellas.set_offsets(_empty_offsets())
    sc_nucleo.set_offsets(_empty_offsets())
    sc_traza_estrellas.set_offsets(_empty_offsets())
    sc_traza_nucleo.set_offsets(_empty_offsets())
    return sc_estrellas, sc_nucleo, sc_traza_estrellas, sc_traza_nucleo

def update(frame):
    actual_estrellas = tray_g1[frame, idx_estrellas, :2]
    actual_nucleo = tray_g1[frame, idx_nucleo, :2]

    i0 = max(0, frame - trail_length)
    traza_estrellas = tray_g1[i0:frame + 1, idx_estrellas, :2].reshape(-1, 2)
    traza_nucleo = tray_g1[i0:frame + 1, idx_nucleo, :2].reshape(-1, 2)

    sc_estrellas.set_offsets(actual_estrellas)
    sc_nucleo.set_offsets(actual_nucleo)
    sc_traza_estrellas.set_offsets(traza_estrellas if traza_estrellas.size else _empty_offsets())
    sc_traza_nucleo.set_offsets(traza_nucleo if traza_nucleo.size else _empty_offsets())

    return sc_estrellas, sc_nucleo, sc_traza_estrellas, sc_traza_nucleo

anim_g1 = FuncAnimation(
    fig,
    update,
    frames=tray_g1.shape[0],
    init_func=init,
    interval=30,
    blit=True
)

plt.close(fig)
display(HTML(anim_g1.to_jshtml()))

In [ ]:
def absorber_particulas(sistema_obj, R_abs, idx_central=None, sumar_masa=True, verbose=True):
    """Elimina particulas dentro del radio de absorcion alrededor del cuerpo central."""
    if sistema_obj.num_particles == 0:
        return []

    if idx_central is None:
        if len(sistema_obj.mass_indices) > 0:
            idx_central = int(sistema_obj.mass_indices[0])
        else:
            idx_central = 0

    central = sistema_obj.particles[idx_central]
    r_c = central["position"]

    nuevas_particles = []
    indices_absorbidos = []

    for i, body in enumerate(sistema_obj.particles):
        if i == idx_central:
            nuevas_particles.append(body)
            continue

        dist = np.linalg.norm(body["position"] - r_c)

        if dist < R_abs:
            indices_absorbidos.append(i)
            if sumar_masa:
                central["mass"] += body["mass"]
            if verbose:
                print(f"Particula {i} absorbida (dist={dist:.4f})")
        else:
            nuevas_particles.append(body)

    # Sincroniza atributos internos para que integrate() siga funcionando
    sistema_obj.particles = nuevas_particles
    sistema_obj.num_particles = len(nuevas_particles)
    sistema_obj.mass = sum(p["mass"] for p in nuevas_particles)

    if sistema_obj.num_particles > 0:
        sistema_obj.masses = np.array([p["mass"] for p in nuevas_particles], dtype=float)
        sistema_obj.positions = np.array([p["position"] for p in nuevas_particles], dtype=float)
        sistema_obj.velocities = np.array([p["velocity"] for p in nuevas_particles], dtype=float)
        sistema_obj.mass_indices = np.where(sistema_obj.masses > 0)[0]

        v_prom = np.mean(np.linalg.norm(sistema_obj.velocities, axis=1))
        n = sistema_obj.num_particles
        sistema_obj.t_relax = (n / (8 * np.log(n))) * (sistema_obj.mass / v_prom**3) if (n > 1 and v_prom > 0) else np.inf
    else:
        sistema_obj.masses = np.array([], dtype=float)
        sistema_obj.positions = np.empty((0, 3), dtype=float)
        sistema_obj.velocities = np.empty((0, 3), dtype=float)
        sistema_obj.mass_indices = np.array([], dtype=int)
        sistema_obj.t_relax = np.inf

    return indices_absorbidos


# Ejemplo de uso:
# absorbidas = absorber_particulas(galaxia1, R_abs=1.0, idx_central=0, sumar_masa=True)
# print(f"Absorbidas: {len(absorbidas)} | Particulas restantes: {galaxia1.num_particles}")

In [ ]:
# =========================
# GRAFICA (trayectorias 2D)
# =========================
import numpy as np
import matplotlib.pyplot as plt

# Simula (ajusta dt y T si quieres)
dt = 0.01
T = 6.0
tray_g1 = galaxia1.integrate(dt=dt, T=T, actualizar=False)

masses = galaxia1.masses
idx_estrellas = np.where(masses == 0)[0]
idx_nucleo = np.where(masses > 0)[0]

plt.figure(figsize=(7, 7))
for i in idx_estrellas:
    plt.plot(tray_g1[:, i, 0], tray_g1[:, i, 1], color="orange", alpha=0.15, linewidth=0.6)

for i in idx_nucleo:
    plt.plot(tray_g1[:, i, 0], tray_g1[:, i, 1], color="gold", alpha=0.9, linewidth=2.0)

plt.scatter(tray_g1[0, idx_estrellas, 0], tray_g1[0, idx_estrellas, 1], s=2, c="cyan", alpha=0.5, label="Estrellas (inicio)")
plt.scatter(tray_g1[0, idx_nucleo, 0], tray_g1[0, idx_nucleo, 1], s=70, c="gold", label="Nucleo (inicio)")

plt.title("Galaxia 1 - Trayectorias")
plt.style.use("dark_background")
plt.xlabel("X")
plt.ylabel("Y")
plt.axis("equal")
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Circle
from IPython.display import HTML, display

# Si la animación pesa mucho al incrustarla en el notebook:
plt.rcParams["animation.embed_limit"] = 40  # MB


def paso_dinamica_simple(sistema_obj, dt):
    """
    Avanza 1 paso temporal (aprox. tipo Euler) usando la aceleración
    gravitatoria definida en tu clase Sistema.
    """
    if sistema_obj.num_particles == 0:
        return

    a = sistema_obj.get_accelerations(
        sistema_obj.positions,
        sistema_obj.masses,
        sistema_obj.mass_indices,
        sistema_obj.num_particles
    )

    sistema_obj.velocities = sistema_obj.velocities + a * dt
    sistema_obj.positions = sistema_obj.positions + sistema_obj.velocities * dt

    # Sincroniza diccionarios internos
    for i in range(sistema_obj.num_particles):
        sistema_obj.particles[i]["position"] = sistema_obj.positions[i]
        sistema_obj.particles[i]["velocity"] = sistema_obj.velocities[i]


def simular_con_absorcion(sistema_obj, dt, T, R_abs, idx_central=0, sumar_masa=True):
    """
    Simula y absorbe partículas en cada paso.
    Requiere que ya tengas definida absorber_particulas(...).
    """
    n_steps = int(T / dt)

    vivos_xy = []
    absorbidos_xy_step = []
    absorbidos_xy_acum = []
    masa_central = []
    n_abs_step = []
    n_abs_acum = []
    tiempos = []

    absorbidos_acumulados_lista = []
    total_abs = 0

    for k in range(n_steps):
        # Guardamos estado previo para saber qué posiciones se absorbieron en este paso
        pos_pre = sistema_obj.positions.copy()

        # Avanza dinámica
        paso_dinamica_simple(sistema_obj, dt)

        # Absorción
        idx_abs = absorber_particulas(
            sistema_obj,
            R_abs=R_abs,
            idx_central=idx_central,
            sumar_masa=sumar_masa,
            verbose=False
        )

        # Posiciones absorbidas (tomadas del estado previo al filtrado)
        if len(idx_abs) > 0:
            abs_xy = pos_pre[idx_abs, :2]
            absorbidos_acumulados_lista.append(abs_xy)
        else:
            abs_xy = np.empty((0, 2))

        if len(absorbidos_acumulados_lista) > 0:
            abs_acum_xy = np.vstack(absorbidos_acumulados_lista)
        else:
            abs_acum_xy = np.empty((0, 2))

        total_abs += len(idx_abs)

        vivos_xy.append(sistema_obj.positions[:, :2].copy())
        absorbidos_xy_step.append(abs_xy)
        absorbidos_xy_acum.append(abs_acum_xy)
        masa_central.append(sistema_obj.particles[idx_central]["mass"])
        n_abs_step.append(len(idx_abs))
        n_abs_acum.append(total_abs)
        tiempos.append((k + 1) * dt)

    return {
        "t": np.array(tiempos),
        "vivos_xy": vivos_xy,
        "abs_step_xy": absorbidos_xy_step,
        "abs_acum_xy": absorbidos_xy_acum,
        "masa_central": np.array(masa_central),
        "n_abs_step": np.array(n_abs_step),
        "n_abs_acum": np.array(n_abs_acum),
        "R_abs": R_abs
    }


def graficar_absorcion(hist):
    """
    Gráficas:
    1) partículas absorbidas por paso y acumuladas
    2) masa del núcleo central en el tiempo
    """
    t = hist["t"]

    fig, axs = plt.subplots(1, 2, figsize=(12, 4))

    axs[0].plot(t, hist["n_abs_step"], label="Absorbidas por paso")
    axs[0].plot(t, hist["n_abs_acum"], label="Absorbidas acumuladas")
    axs[0].set_xlabel("Tiempo")
    axs[0].set_ylabel("Número de partículas")
    axs[0].set_title("Absorción de partículas")
    axs[0].legend()

    axs[1].plot(t, hist["masa_central"], color="darkred")
    axs[1].set_xlabel("Tiempo")
    axs[1].set_ylabel("Masa del núcleo")
    axs[1].set_title("Evolución de masa central")

    plt.tight_layout()
    plt.show()


def animar_absorcion(hist, intervalo=40):
    """
    Animación:
    - cian: partículas vivas
    - rojo: partículas absorbidas acumuladas
    - círculo blanco: radio de absorción
    """
    vivos_xy = hist["vivos_xy"]
    abs_acum_xy = hist["abs_acum_xy"]
    R_abs = hist["R_abs"]

    # Escala visual
    max_r = 0.0
    for arr in vivos_xy:
        if arr.size > 0:
            max_r = max(max_r, np.max(np.linalg.norm(arr, axis=1)))
    for arr in abs_acum_xy:
        if arr.size > 0:
            max_r = max(max_r, np.max(np.linalg.norm(arr, axis=1)))
    lim = 1.2 * max(max_r, R_abs * 2)

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.set_facecolor("black")
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    ax.set_title("Absorción de partículas")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")

    sc_vivas = ax.scatter([], [], s=2, c="cyan", alpha=0.8, label="Vivas")
    sc_abs = ax.scatter([], [], s=6, c="red", alpha=0.8, label="Absorbidas")
    circ = Circle((0, 0), R_abs, edgecolor="white", facecolor="none", linestyle="--", linewidth=1.2)
    ax.add_patch(circ)
    ax.legend(loc="upper right")

    def _empty():
        return np.empty((0, 2))

    def init():
        sc_vivas.set_offsets(_empty())
        sc_abs.set_offsets(_empty())
        return sc_vivas, sc_abs, circ

    def update(frame):
        vivos = vivos_xy[frame]
        abso = abs_acum_xy[frame]

        sc_vivas.set_offsets(vivos if vivos.size else _empty())
        sc_abs.set_offsets(abso if abso.size else _empty())
        return sc_vivas, sc_abs, circ

    anim = FuncAnimation(
        fig, update,
        frames=len(vivos_xy),
        init_func=init,
        interval=intervalo,
        blit=True
    )

    plt.close(fig)
    display(HTML(anim.to_jshtml()))
    return anim




In [ ]:
galaxia_abs = Galaxia(
    num_estrellas=1000,
    masa_nucleo=1e3,
    radio_max=10.0,
    radio_min=1.5   # antes 3
)

# Reducimos un poco velocidades de estrellas (no el núcleo)
factor_vel = 0.9
galaxia_abs.velocities[1:] *= factor_vel
for i in range(1, galaxia_abs.num_particles):
    galaxia_abs.particles[i]["velocity"] = galaxia_abs.velocities[i]

hist_abs = simular_con_absorcion(
    galaxia_abs,
    dt=0.01,
    T=12.0,      # antes 6
    R_abs=1.2,   # antes 1.0
    idx_central=0,
    sumar_masa=True
)

graficar_absorcion(hist_abs)
anim = animar_absorcion(hist_abs, intervalo=35)